# Wrangling Project
## Scrapping Spotrac Data
#### Drew Peterson
***

In [15]:
#Imports
import pandas as pd
import requests

In [16]:
#Scrapes data using requests package and html_
url = "https://www.spotrac.com/mlb/cash/_/year/2025"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
html = response.text

#Parses tables
tables = pd.read_html(html)

spotrac_df = tables[0]

display(spotrac_df)

/var/folders/x5/cndptcpn0bngcgpkcg_sbzkr0000gn/T/ipykernel_20724/118092272.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


,Rank,Team,Record,Signed,Avg Age Team,Active,Retained,Total Cash
0,1.0,NYM NYM,83-79,28.0,29.3,"$312,899,806",$0,"$400,965,292"
1,2.0,LAD LAD,93-69,27.0,29.6,"$293,126,821",$0,"$393,082,178"
2,3.0,NYY NYY,94-68,28.0,29.7,"$205,097,386","$5,000,000","$323,561,450"
3,4.0,PHI PHI,96-66,27.0,30.3,"$226,300,409","$3,000,000","$308,208,429"
4,5.0,TOR TOR,94-68,28.0,28.9,"$179,210,208",$0,"$290,944,111"
5,6.0,HOU HOU,87-75,28.0,28.6,"$135,180,023","$701,520","$253,425,770"
6,7.0,TEX TEX,81-81,29.0,30.1,"$113,977,024",$0,"$251,697,316"
7,8.0,ATL ATL,76-86,28.0,29.5,"$128,648,209","$3,500,000","$237,076,106"
8,9.0,SD SD,90-72,28.0,29.3,"$160,473,830","$6,300,000","$236,031,133"
9,10.0,CHC CHC,92-70,28.0,29.6,"$173,037,007","$2,750,000","$231,356,720"


In [17]:
#Selects columns that I want to keep
spotrac_cleaned = spotrac_df[["Team", "Active", "Total Cash"]]
display(spotrac_cleaned)

,Team,Active,Total Cash
0,NYM NYM,"$312,899,806","$400,965,292"
1,LAD LAD,"$293,126,821","$393,082,178"
2,NYY NYY,"$205,097,386","$323,561,450"
3,PHI PHI,"$226,300,409","$308,208,429"
4,TOR TOR,"$179,210,208","$290,944,111"
5,HOU HOU,"$135,180,023","$253,425,770"
6,TEX TEX,"$113,977,024","$251,697,316"
7,ATL ATL,"$128,648,209","$237,076,106"
8,SD SD,"$160,473,830","$236,031,133"
9,CHC CHC,"$173,037,007","$231,356,720"


In [18]:
#Drops rows 30 and 31
spotrac_cleaned = spotrac_cleaned.iloc[:-2]
display(spotrac_cleaned)

,Team,Active,Total Cash
0,NYM NYM,"$312,899,806","$400,965,292"
1,LAD LAD,"$293,126,821","$393,082,178"
2,NYY NYY,"$205,097,386","$323,561,450"
3,PHI PHI,"$226,300,409","$308,208,429"
4,TOR TOR,"$179,210,208","$290,944,111"
5,HOU HOU,"$135,180,023","$253,425,770"
6,TEX TEX,"$113,977,024","$251,697,316"
7,ATL ATL,"$128,648,209","$237,076,106"
8,SD SD,"$160,473,830","$236,031,133"
9,CHC CHC,"$173,037,007","$231,356,720"


In [19]:
#Cleans Team abbreviation column
spotrac_cleaned["Team"] = spotrac_cleaned["Team"].str.split().str[0]

#Removes $ sign from Active and Total Cash columns
cols = ["Active", "Total Cash"]

for col in cols:
    spotrac_cleaned[col] = (
        spotrac_cleaned[col]
        .replace(r"[\$,]", "", regex=True)
        .astype(float)
    )
print(spotrac_cleaned.dtypes)
display(spotrac_cleaned)

Team           object
Active        float64
Total Cash    float64
dtype: object


,Team,Active,Total Cash
0,NYM,312899806.0,400965292.0
1,LAD,293126821.0,393082178.0
2,NYY,205097386.0,323561450.0
3,PHI,226300409.0,308208429.0
4,TOR,179210208.0,290944111.0
5,HOU,135180023.0,253425770.0
6,TEX,113977024.0,251697316.0
7,ATL,128648209.0,237076106.0
8,SD,160473830.0,236031133.0
9,CHC,173037007.0,231356720.0


In [20]:
#Reads Baseball_Reference_Cleaned.csv, created in different notebook
br_df = pd.read_csv("Baseball_Reference_Cleaned.csv", index_col=0)
display(br_df)


,Rank,Team Name,Wins,Losses,Runs Allowed
0,1,Milwaukee Brewers,97,65,3.9
1,2,Philadelphia Phillies,96,66,4.0
2,3,Toronto Blue Jays,94,68,4.5
3,4,New York Yankees,94,68,4.2
4,5,Los Angeles Dodgers,93,69,4.2
5,6,Chicago Cubs,92,70,4.0
6,7,Seattle Mariners,90,72,4.3
7,8,San Diego Padres,90,72,3.8
8,9,Boston Red Sox,89,73,4.2
9,10,Cleveland Guardians,88,74,4.0


In [21]:
#maps Team abbreviation column named Team for Baseball Reference dataframe
team_map = {
    "Milwaukee Brewers": "MIL",
    "Philadelphia Phillies": "PHI",
    "Toronto Blue Jays": "TOR",
    "New York Yankees": "NYY",
    "Los Angeles Dodgers": "LAD",
    "Chicago Cubs": "CHC",
    "Seattle Mariners": "SEA",
    "San Diego Padres": "SD",
    "Boston Red Sox": "BOS",
    "Cleveland Guardians": "CLE",
    "Houston Astros": "HOU",
    "Detroit Tigers": "DET",
    "New York Mets": "NYM",
    "Cincinnati Reds": "CIN",
    "Kansas City Royals": "KC",
    "San Francisco Giants": "SF",
    "Texas Rangers": "TEX",
    "Arizona Diamondbacks": "ARI",
    "Miami Marlins": "MIA",
    "St. Louis Cardinals": "STL",
    "Tampa Bay Rays": "TB",
    "Athletics": "ATH",
    "Atlanta Braves": "ATL",
    "Baltimore Orioles": "BAL",
    "Los Angeles Angels": "LAA",
    "Pittsburgh Pirates": "PIT",
    "Minnesota Twins": "MIN",
    "Washington Nationals": "WSH",
    "Chicago White Sox": "CHW",
    "Colorado Rockies": "COL"
}

#Applies mapping
br_df["Team"] = br_df["Team Name"].map(team_map)
display(br_df)

,Rank,Team Name,Wins,Losses,Runs Allowed,Team
0,1,Milwaukee Brewers,97,65,3.9,MIL
1,2,Philadelphia Phillies,96,66,4.0,PHI
2,3,Toronto Blue Jays,94,68,4.5,TOR
3,4,New York Yankees,94,68,4.2,NYY
4,5,Los Angeles Dodgers,93,69,4.2,LAD
5,6,Chicago Cubs,92,70,4.0,CHC
6,7,Seattle Mariners,90,72,4.3,SEA
7,8,San Diego Padres,90,72,3.8,SD
8,9,Boston Red Sox,89,73,4.2,BOS
9,10,Cleveland Guardians,88,74,4.0,CLE


In [28]:
#Merges dataframes
merged_df = pd.merge(br_df, spotrac_cleaned, on="Team", how="inner")

print(merged_df.shape)
display(merged_df)

(30, 8)


,Rank,Team Name,Wins,Losses,Runs Allowed,Team,Active,Total Cash
0,1,Milwaukee Brewers,97,65,3.9,MIL,84095680.0,140266827.0
1,2,Philadelphia Phillies,96,66,4.0,PHI,226300409.0,308208429.0
2,3,Toronto Blue Jays,94,68,4.5,TOR,179210208.0,290944111.0
3,4,New York Yankees,94,68,4.2,NYY,205097386.0,323561450.0
4,5,Los Angeles Dodgers,93,69,4.2,LAD,293126821.0,393082178.0
5,6,Chicago Cubs,92,70,4.0,CHC,173037007.0,231356720.0
6,7,Seattle Mariners,90,72,4.3,SEA,131472218.0,193414894.0
7,8,San Diego Padres,90,72,3.8,SD,160473830.0,236031133.0
8,9,Boston Red Sox,89,73,4.2,BOS,122765355.0,229129601.0
9,10,Cleveland Guardians,88,74,4.0,CLE,50354620.0,118369450.0


In [30]:
#Moves columns for better readability
merged_df = merged_df[["Team", "Team Name", "Rank","Wins","Losses","Runs Allowed","Active","Total Cash"]]
display(merged_df.head())

,Team,Team Name,Rank,Wins,Losses,Runs Allowed,Active,Total Cash
0,MIL,Milwaukee Brewers,1,97,65,3.9,84095680.0,140266827.0
1,PHI,Philadelphia Phillies,2,96,66,4.0,226300409.0,308208429.0
2,TOR,Toronto Blue Jays,3,94,68,4.5,179210208.0,290944111.0
3,NYY,New York Yankees,4,94,68,4.2,205097386.0,323561450.0
4,LAD,Los Angeles Dodgers,5,93,69,4.2,293126821.0,393082178.0


In [32]:
#Saves as csv for my analysis notebook
merged_df.to_csv("salary_analysis_merged_df.csv")